# Useful Scripts

## Dataset creation

### Extracting and consolidating .json files

This script extracts the donations from the json files, and consolidates into one tabular dataset.

To use this script, you need to provide:
* The folder (relative path) where the .json files are stored
* The source type: for donated data, the information that comes after 'source=' in the filename
* The key that you want to extract
* The folder (relative path) where you want to store the dataset
* The filename for the dataset

In [ ]:
import json, os
import pandas as pd

In [ ]:
def extract_consolidate_donations(input_path, source, key, output_path, output_filename):
        files = os.listdir(input_path)
        files = [f for f in files if f.endswith('.json') and source in f]
        print('found', len(files), 'files')


        results = pd.DataFrame()
        errors = []

        for file in files:
            try:
                print('processing', file)
                with open(os.path.join(input_path, file), 'r') as f:
                    tmp_json = json.load(f)
                    for item in tmp_json:
                        if key in item.keys():
                            donations = item[key]
                            tmp_df = pd.DataFrame(item[key])
                            tmp_df['filename'] = file
                            tmp_df['file_creation_date'] = file.split('_')[0]
                            tmp_df['participant'] = file.split('participant=')[1].split('_source')[0]
                            results = pd.concat([results, tmp_df], ignore_index=True)
            except Exception as e:
                print('Error processing', file, ':', e)
                errors.append({'filename': file, 'error': str(e)})

        results.to_excel(os.path.join(output_path, output_filename + '.xlsx'), index=False)
        if len(errors) > 0:
            pd.DataFrame(errors).to_excel(os.path.join(output_path, output_filename + '_errors.xlsx'), index=False)

        print('saved', os.path.join(output_path, output_filename + '.xlsx'))

        return

In [ ]:
input_path = ''
source = ''
key = ''
output_path = ''
output_filename = ''


In [ ]:
extract_consolidate_donations(input_path, source, key, output_path, output_filename)

### Extracting and consolidating other user logging data 

This script creates files with:
1. The number of user omissions in the donation per participant
2. The logs of the donation in a tabular format

You need to provide:
* The folder (relative path) where the .json files are stored
* The source type: for donated data, the information that comes after 'source=' in the filename
* The folder (relative path) where you want to store the logs



In [ ]:
import json, os
import pandas as pd

In [ ]:
def extract_consolidate_logs(input_path, source, output_path):
        files = os.listdir(input_path)
        files = [f for f in files if f.endswith('.json')]
        print('found', len(files), 'files')


        results = pd.DataFrame()
        key = 'user_omissions'
        logs = pd.DataFrame()
        errors = []

        for file in files:
            try:
                print('processing', file)
                if source in file:
                    with open(os.path.join(input_path, file), 'r') as f:
                        tmp_json = json.load(f)
                        for item in tmp_json:
                            if key in item.keys():
                                tmp_omissions = {}
                                tmp_omissions['filename'] = file
                                tmp_omissions['file_creation_date'] = file.split('_')[0]
                                tmp_omissions['participant'] = file.split('participant=')[1].split('_source')[0]
                                tmp_omissions['user_omissions'] = item[key]
                                tmp_df = pd.DataFrame([tmp_omissions])

                                results = pd.concat([results, tmp_df], ignore_index=True)

                if 'tracking' in file:
                    tmp_tracking = []
                    with open(os.path.join(input_path, file), 'r') as f:
                        tmp_json = json.load(f)
                        for log_line in tmp_json:
                            if '---' in log_line:
                                tmp_log = {}
                                tmp_log['filename'] = file
                                tmp_log['file_creation_date'] = file.split('_')[0]
                                tmp_log['participant'] = file.split('participant=')[1].split('_source')[0] 
                                
                                log_line = log_line.split('---')
                                tmp_log['log_timestamp'] = log_line[0].strip()
                                tmp_log['log_level'] = log_line[1].strip()
                                tmp_log['log_event'] = log_line[2].strip()
                                tmp_log['log_message'] = log_line[3].strip()
                                tmp_tracking.append(tmp_log)
                    
                    logs = pd.concat([logs, pd.DataFrame(tmp_tracking)], ignore_index=True)
                            
            except Exception as e:
                print('Error processing', file, ':', e)
                errors.append({'filename': file, 'error': str(e)})

        results.to_excel(os.path.join(output_path, 'user_omissions_'+ source + '.xlsx'), index=False)
        logs.to_excel(os.path.join(output_path, 'tracking.xlsx'), index=False)
        if len(errors) > 0:
            pd.DataFrame(errors).to_excel(os.path.join(output_path, 'log_errors.xlsx'), index=False)

        

        return

In [ ]:
input_path = ''
source = ''
output_path = ''


In [ ]:
extract_consolidate_logs(input_path, source, output_path)